In [ ]:
import os
import pickle
import pandas as pd
import numpy as np

# =====================================================================
# 1. OPTIMIZED CORE FUNCTIONS
# =====================================================================
def ComputeErrors(df, id_col, neut_col, blast_col, filtered_sims):
    """
    Computes Euclidean distance error between clinical targets and ODE simulations.
    Pre-filters missing values and minimizes dictionary overhead.
    """
    # 1. Drop rows with missing values to avoid NaN propagation
    df_clean = df.dropna(subset=[id_col, blast_col, neut_col]).copy()
    
    patient_ids = df_clean[id_col].astype(str).str.strip().values
    target_blasts = df_clean[blast_col].to_numpy(dtype=float)
    target_neuts = (df_clean[neut_col].to_numpy(dtype=float) * 5 * 10**9 / 80)
    
    neut_range = 4.0 * 10**8
    
    sim_blasts = []
    sim_neuts = []
    sim_times = []
    sim_phsc = []
    sim_coef = []
    
    for df_sim in filtered_sims:
        sim_blasts.append(df_sim['BlastPerc'].values * 100.0)
        sim_neuts.append(df_sim['Neut'].values)
        sim_times.append(df_sim['Time'].values)
        sim_phsc.append(df_sim['pHSC'].iloc[0])
        sim_coef.append(df_sim['Coef'].iloc[0])
        
    results_list = []
    n_sims = len(filtered_sims)
    
    # 3. Compute best fit per patient-simulation pair
    for p_id, t_blast, t_neut in zip(patient_ids, target_blasts, target_neuts):
        for j in range(n_sims):
            b_arr = sim_blasts[j]
            n_arr = sim_neuts[j]
            
            blast_err = (b_arr - t_blast) / 100.0
            neutro_err = (n_arr - t_neut) / neut_range
            
            combined_err_sq = (blast_err**2) + (neutro_err**2)
            idx_min = np.argmin(combined_err_sq)
            
            results_list.append({
                'ID': p_id,
                'pHSC': sim_phsc[j],
                'coef': sim_coef[j],
                'sim_day': sim_times[j][idx_min],
                'err_blast': blast_err[idx_min],
                'err_neutro': neutro_err[idx_min],
                'total_err': float(np.sqrt(combined_err_sq[idx_min]))
            })
            
    return results_list

def GetMinimumErrors(results):
    """Isolates the single best absolute fit across all simulations for each patient."""
    df_results = pd.DataFrame(results)
    if df_results.empty:
        return df_results
    idx_min = df_results.groupby('ID')['total_err'].idxmin()
    return df_results.loc[idx_min].reset_index(drop=True)

def compute_weighted_parameters_dynamic(df_all_simulations, accuracy_tolerance=0.1):
    """
    Returns the personalized dynamic inverse-error weighted average for each patient.
    """
    if not isinstance(df_all_simulations, pd.DataFrame):
        df_all_simulations = pd.DataFrame(df_all_simulations)

    df_all_simulations["ID"] = df_all_simulations["ID"].astype(str).str.strip()
    weighted_fits_list = []
    
    for p_id, group in df_all_simulations.groupby("ID"):
        min_err = group["total_err"].min()
        if np.isnan(min_err):
            continue
            
        max_allowed_error = min_err * (1.0 + accuracy_tolerance)
        
        top_fits = group[group["total_err"] <= max_allowed_error]
        if top_fits.empty:
            top_fits = group.nsmallest(1, "total_err")
            
        pHSC_vals = top_fits["pHSC"].to_numpy()
        coef_vals = top_fits["coef"].to_numpy()
        err_vals = top_fits["total_err"].to_numpy()
        
        weights = 1.0 / (err_vals + 1e-12)
        norm_weights = weights / weights.sum()
        
        weighted_fits_list.append({
            "ID": p_id,
            "pHSC_weighted": float(np.sum(pHSC_vals * norm_weights)),
            "coef_weighted": float(np.sum(coef_vals * norm_weights)),
            "err_weighted": float(np.sum(err_vals * norm_weights)),
            "min_error_baseline": float(min_err),
            "accepted_cloud_size": int(len(top_fits))
        })

    return pd.DataFrame(weighted_fits_list)

# =====================================================================
# 2. DATA LOADING & UNIFIED EXECUTION
# =====================================================================
if __name__ == "__main__":
    os.makedirs("IntermediateResults", exist_ok=True)

    print("Loading trajectory data...")
    with open("IntermediateResults/venex_best_gamma_runs.pkl", "rb") as f1:
        venex_best_gamma_runs = pickle.load(f1)
    with open("IntermediateResults/moffitt_best_gamma_runs.pkl", "rb") as f2:
        moffitt_best_gamma_runs = pickle.load(f2)

    flat_venex_sims = [df for item in venex_best_gamma_runs for df in (item if isinstance(item, list) else [item])]
    flat_moffitt_sims = [df for item in moffitt_best_gamma_runs for df in (item if isinstance(item, list) else [item])]

    # ---------------------------------------------------------
    # VENEX COHORT PROCESSING
    # ---------------------------------------------------------
    print("Computing errors for Unified VenEx Cohort...")
    venex_cohort = pd.read_excel("InputData/venex_cohort_OS_labs_with_mutations.xlsx")
    venex_errors = ComputeErrors(venex_cohort, 'Study_subject', 'b_neut', 'bm_blasts', flat_venex_sims)
    
    with open('IntermediateResults/venex_allerrors.pkl', 'wb') as file:
        pickle.dump(venex_errors, file, protocol=pickle.HIGHEST_PROTOCOL)
        
    venex_min = GetMinimumErrors(venex_errors)
    venex_weighted = compute_weighted_parameters_dynamic(venex_errors, accuracy_tolerance=0.1)
    
    venex_min.to_csv("IntermediateResults/venex_minimum_errors.csv", index=False)
    venex_weighted.to_csv("IntermediateResults/venex_weighted_params.csv", index=False)

    # ---------------------------------------------------------
    # MOFFITT COHORT PROCESSING
    # ---------------------------------------------------------
    print("Computing errors for Unified Moffitt Cohort...")
    moffit_cohort = pd.read_excel("InputData/moffitt_cohort_OS_labs.xlsx")
    moffit_errors = ComputeErrors(moffit_cohort, 'Subject_ID', 'ANC at Dx', 'BM Blasts at Dx by flow', flat_moffitt_sims)
    
    with open('IntermediateResults/moffitt_allerrors.pkl', 'wb') as file:
        pickle.dump(moffit_errors, file, protocol=pickle.HIGHEST_PROTOCOL)

    moffitt_min = GetMinimumErrors(moffit_errors)
    moffitt_weighted = compute_weighted_parameters_dynamic(moffit_errors, accuracy_tolerance=0.1)
    
    moffitt_min.to_csv("IntermediateResults/moffitt_minimum_errors.csv", index=False)
    moffitt_weighted.to_csv("IntermediateResults/moffitt_weighted_params.csv", index=False)

    print("\nAll errors successfully computed, aggregated, and saved!")